# Wearable Fatigue ML V2：从原始压缩包到 Demo 的单 Notebook 完整复现

这是本项目唯一需要重点学习的总流程 Notebook。它假设你最开始只有原始数据集压缩包 `WSD4FEDSRM.zip`，并把最终项目的实现代码全部展开在本文件中。

本 Notebook **不会调用** `scripts/*.py`、`src/*.py` 或其他 Notebook。除 Python 第三方库和原始/生成数据文件外，所有关键函数、参数、模型与评估逻辑都直接写在下面的代码单元格里。

完整路线：

1. 检查或解压原始数据集。
2. 读取 README、description、Borg RPE 和 Upper-arm ACC/Gyroscope。
3. 按 100 Hz、10 秒非重叠窗口提取 56 个时域特征。
4. 构建并检查 `fatigue_features.csv`。
5. 用 subject-level GroupKFold 做回归与分类 baseline。
6. 回归与分类都用 Nested GroupKFold 做正式调参和外层 OOF 评估。
7. 对最佳分类器在训练受试者内部做 F2 阈值优化。
8. 生成最终 OOF 表、图、Demo 数据与 Streamlit 程序。

项目边界保持不变：仅使用 Upper-arm ACC + Gyroscope；不使用 EMG、PPG、磁力计或其他佩戴位置；不修改 RPE 标签；不做深度学习。

## 1. 导入依赖

这一格集中导入整个流程需要的库。后面不会再从项目中的 `.py` 文件导入函数。

- `pandas/numpy`：表格和数值计算。
- `matplotlib`：结果图。
- `scikit-learn`：GroupKFold、模型、调参和指标。
- `xgboost`：XGBoost 回归与分类。
- `zipfile/pathlib`：压缩包和路径处理。

在 Anaconda 中复现前，应确认当前 Jupyter kernel 已安装 `xgboost`。

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json
import re
import sys
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBClassifier, XGBRegressor

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("Python:", sys.version)
print("pandas:", pd.__version__)

## 2. 设置唯一需要手动修改的项目路径

从仓库根目录或 notebooks 目录启动 Jupyter 时，项目路径会自动定位。将 WSD4FEDSRM.zip 放在仓库的上一级目录。

正式参数也集中放在这一格：随机种子 42、外层 5 折、内层 4 折、随机搜索 20 次。

In [ ]:
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
PROJECT_PARENT = PROJECT_DIR.parent
RAW_ZIP = PROJECT_PARENT / "WSD4FEDSRM.zip"
RAW_DATA_ROOT = PROJECT_PARENT / "WSD4FEDSRM"

DATA_DIR = PROJECT_DIR / "data_processed"
RESULTS_DIR = PROJECT_DIR / "results" / "resume_final"
FIGURES_DIR = RESULTS_DIR / "figures"

FATIGUE_FEATURES_CSV = DATA_DIR / "fatigue_features.csv"
EXCLUDED_WINDOWS_CSV = DATA_DIR / "excluded_windows.csv"
DEMO_FINAL_CSV = DATA_DIR / "demo_predictions_final.csv"

RANDOM_STATE = 42
N_SPLITS_OUTER = 5
N_SPLITS_INNER = 4
N_ITER_RANDOM_SEARCH = 20

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {
    "PROJECT_DIR": PROJECT_DIR,
    "RAW_ZIP": RAW_ZIP,
    "RAW_DATA_ROOT": RAW_DATA_ROOT,
    "DATA_DIR": DATA_DIR,
    "RESULTS_DIR": RESULTS_DIR,
}.items():
    print(f"{name}: {path} | exists={path.exists()}")

## 3. 从压缩包准备原始数据

如果 `WSD4FEDSRM` 文件夹已经存在，这一格只检查路径，不重复解压。如果文件夹不存在但压缩包存在，则解压到项目上一级目录。原始 CSV 不会被修改。

In [ ]:
if not RAW_DATA_ROOT.exists():
    if not RAW_ZIP.exists():
        raise FileNotFoundError(
            f"既没有找到解压目录 {RAW_DATA_ROOT}，也没有找到压缩包 {RAW_ZIP}。"
        )
    print("正在解压原始数据集，请等待……")
    with ZipFile(RAW_ZIP, "r") as archive:
        archive.extractall(PROJECT_PARENT)

assert RAW_DATA_ROOT.exists(), f"解压后仍未找到数据集目录: {RAW_DATA_ROOT}"
print("原始数据集目录已就绪:", RAW_DATA_ROOT)

## 4. 阅读 README 与 description

真正做数据项目前，先看数据集作者给出的说明，而不是直接猜目录含义。我们会据此核对 IMU 采样率、Borg 时间间隔和六个 task 的含义。

In [ ]:
def read_text_flexibly(path: Path) -> str:
    for encoding in ("utf-8", "utf-8-sig", "gb18030", "latin-1"):
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue
    return path.read_text(encoding="utf-8", errors="replace")


readme_text = read_text_flexibly(RAW_DATA_ROOT / "README.txt")
description_text = read_text_flexibly(RAW_DATA_ROOT / "description.txt")

print("===== README.txt =====")
print(readme_text)
print("\n===== description.txt =====")
print(description_text)

## 5. 固定已经验证的数据定义

这一格是整个项目的数据合同：100 Hz、10 秒非重叠窗口、每通道 1000 点，以及 Borg task 与真实运动文件夹的映射。原始 ACC/GYR 列名也必须严格符合预期。

In [ ]:
SAMPLING_RATE_HZ = 100
WINDOW_SECONDS = 10
WINDOW_SIZE = SAMPLING_RATE_HZ * WINDOW_SECONDS

BORG_CSV = RAW_DATA_ROOT / "Borg data" / "borg_data.csv"
IMU_ROOT = RAW_DATA_ROOT / "EMG, IMU, and PPG data"

TASK_TO_EXERCISE = {
    "task1_35i": "30-40_ internal rotation",
    "task2_45i": "40-50_ internal rotation",
    "task3_55i": "50-60_ internal rotation",
    "task4_35e": "30-40_ external rotation",
    "task5_45e": "40-50_ external rotation",
    "task6_55e": "50-60_ external rotation",
}

ACC_RAW_COLUMNS = ["acc_x(m/s^2)", "acc_y(m/s^2)", "acc_z(m/s^2)"]
GYR_RAW_COLUMNS = ["gyr_x(rad/s)", "gyr_y(rad/s)", "gyr_z(rad/s)"]
ACC_COLUMNS = ["acc_x", "acc_y", "acc_z"]
GYR_COLUMNS = ["gyr_x", "gyr_y", "gyr_z"]
SENSOR_COLUMNS = ACC_COLUMNS + GYR_COLUMNS
METADATA_COLUMNS = ["subject", "task", "exercise_folder", "window_end_sec", "rpe"]
STATS = ["mean", "std", "min", "max", "range", "rms", "median"]

assert WINDOW_SIZE == 1000
assert BORG_CSV.exists()
for task, folder in TASK_TO_EXERCISE.items():
    assert (IMU_ROOT / folder).exists(), f"运动文件夹不存在: {folder}"
    print(task, "->", folder)

## 6. 定义 Borg 与 IMU 的读取函数

这里统一 subject 名称、处理 Borg 表中的空白 subject 单元格、识别 `10_sec/20_sec/...` 列，并严格检查 Upper-arm ACC/GYR 的三列格式。函数只读原始文件。

In [ ]:
def normalize_subject(value: object) -> str | None:
    match = re.search(r"(\d+)", str(value))
    if not match:
        return None
    return f"subject_{int(match.group(1))}"


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    output.columns = [str(col).strip() for col in output.columns]
    return output


def load_borg_table(path: Path = BORG_CSV) -> pd.DataFrame:
    borg = normalize_columns(pd.read_csv(path, dtype=str))
    borg["subject"] = borg["subject"].replace(r"^\s*$", pd.NA, regex=True).ffill()
    borg["subject"] = borg["subject"].map(normalize_subject)
    borg["task_order"] = borg["task_order"].astype(str).str.strip()
    borg["length_of_trial_(sec)"] = pd.to_numeric(
        borg["length_of_trial_(sec)"].astype(str).str.strip(), errors="coerce"
    )
    return borg


def borg_time_columns(borg: pd.DataFrame) -> list[tuple[int, str]]:
    output: list[tuple[int, str]] = []
    for col in borg.columns:
        match = re.fullmatch(r"(\d+)_sec", str(col))
        if match:
            output.append((int(match.group(1)), col))
    return sorted(output)


def load_sensor_csv(path: Path, expected_raw_columns: list[str], output_columns: list[str]) -> pd.DataFrame:
    sensor = normalize_columns(pd.read_csv(path))
    if list(sensor.columns) != expected_raw_columns:
        raise ValueError(f"unexpected columns in {path}: {list(sensor.columns)}")
    sensor = sensor.rename(columns=dict(zip(expected_raw_columns, output_columns)))
    for col in output_columns:
        sensor[col] = pd.to_numeric(sensor[col], errors="coerce")
    return sensor


def load_acc(path: Path) -> pd.DataFrame:
    return load_sensor_csv(path, ACC_RAW_COLUMNS, ACC_COLUMNS)


def load_gyr(path: Path) -> pd.DataFrame:
    return load_sensor_csv(path, GYR_RAW_COLUMNS, GYR_COLUMNS)

## 7. 实际检查 Borg 标签结构

这里展示 Borg 表、受试者、任务和时间标签列，确认标签按 10 秒递增。后面仍会逐个检查每个时间点是否真的有 RPE 数值。

In [ ]:
borg = load_borg_table()
time_columns = borg_time_columns(borg)
time_points = [time_sec for time_sec, _ in time_columns]

print("Borg shape:", borg.shape)
print("Borg subjects:", borg["subject"].nunique())
print("Borg tasks:", sorted(borg["task_order"].dropna().unique()))
print("前 20 个时间标签列:", time_columns[:20])
display(borg.head(12))

assert set(TASK_TO_EXERCISE).issubset(set(borg["task_order"].dropna()))
assert all(second - first == 10 for first, second in zip(time_points, time_points[1:]))

## 8. 定义 10 秒窗口边界与有效性检查

例如 `window_end_sec=20` 对应 10–20 秒，Python 行切片是 `[1000:2000]`，恰好 1000 行。窗口含 NaN/inf、长度不足或 Borg 标签缺失时都不会进入训练数据。

In [ ]:
def has_real_rpe(value: object) -> bool:
    return (not pd.isna(value)) and str(value).strip() != ""


def window_bounds(window_end_sec: int, sampling_rate_hz: int = SAMPLING_RATE_HZ) -> tuple[int, int]:
    end_idx = window_end_sec * sampling_rate_hz
    start_idx = end_idx - WINDOW_SIZE
    return start_idx, end_idx


def has_invalid_sensor_value(window: pd.DataFrame) -> bool:
    values = window.to_numpy(dtype=float)
    return bool(np.isnan(values).any() or np.isinf(values).any())


print("20 秒结束窗口的行索引:", window_bounds(20))
assert window_bounds(20) == (1000, 2000)

## 9. 定义 56 个时域特征

6 个原始通道各提取 7 个统计量，得到 42 个特征；ACC 和 GYR 三轴模长各提取 7 个统计量，再得到 14 个，总计 56 个。`std` 使用样本标准差 `ddof=1`。

In [ ]:
def series_features(values: pd.Series, prefix: str) -> dict[str, float]:
    arr = values.to_numpy(dtype=float)
    min_value = float(np.min(arr))
    max_value = float(np.max(arr))
    return {
        f"{prefix}_mean": float(np.mean(arr)),
        f"{prefix}_std": float(np.std(arr, ddof=1)),
        f"{prefix}_min": min_value,
        f"{prefix}_max": max_value,
        f"{prefix}_range": max_value - min_value,
        f"{prefix}_rms": float(np.sqrt(np.mean(arr ** 2))),
        f"{prefix}_median": float(np.median(arr)),
    }


def extract_window_features(window: pd.DataFrame) -> dict[str, float]:
    features: dict[str, float] = {}
    for col in SENSOR_COLUMNS:
        features.update(series_features(window[col], col))
    acc_mag = np.sqrt((window[ACC_COLUMNS].to_numpy(dtype=float) ** 2).sum(axis=1))
    gyr_mag = np.sqrt((window[GYR_COLUMNS].to_numpy(dtype=float) ** 2).sum(axis=1))
    features.update(series_features(pd.Series(acc_mag), "acc_mag"))
    features.update(series_features(pd.Series(gyr_mag), "gyr_mag"))
    return features


def feature_columns() -> list[str]:
    columns: list[str] = []
    for prefix in SENSOR_COLUMNS + ["acc_mag", "gyr_mag"]:
        columns.extend([f"{prefix}_{stat}" for stat in STATS])
    return columns


FEATURE_COLUMNS = feature_columns()
print("特征总数:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)
assert len(FEATURE_COLUMNS) == 56

## 10. 定义从原始 CSV 构建特征表的完整函数
这部分还没有训练模型，它负责生成模型需要的 fatigue_features.csv。最后输出两个表：
features_df：1310 个有效窗口
excluded_df：80 个被排除窗口及原因



遍历 `task → exercise folder → subject`，查找 Borg 行和 Upper-arm ACC/GYR，逐个检查真实 Borg 标签与完整 IMU 窗口，再计算 56 个特征。不合格记录单独保存，不填补标签，也不截断原始文件。

In [ ]:
def subject_dirs(exercise_path: Path) -> list[Path]:
    directories = [path for path in exercise_path.iterdir() if path.is_dir() and normalize_subject(path.name)]
    return sorted(directories, key=lambda path: int(re.search(r"(\d+)", path.name).group(1)))


def excluded_record(subject, task, exercise_folder, window_end_sec, reason, detail="") -> dict[str, Any]:
    return {
        "subject": subject,
        "task": task,
        "exercise_folder": exercise_folder,
        "window_end_sec": window_end_sec,
        "reason": reason,
        "detail": detail,
    }


def build_feature_dataset() -> tuple[pd.DataFrame, pd.DataFrame]:
    borg_table = load_borg_table()
    label_columns = dict(borg_time_columns(borg_table))
    feature_rows: list[dict[str, Any]] = []
    excluded_rows: list[dict[str, Any]] = []

    for task, exercise_folder in TASK_TO_EXERCISE.items():
        exercise_path = IMU_ROOT / exercise_folder
        for subject_dir in subject_dirs(exercise_path):
            subject = normalize_subject(subject_dir.name)
            assert subject is not None
            subject_borg = borg_table[(borg_table["subject"] == subject) & (borg_table["task_order"] == task)]
            if subject_borg.empty:
                excluded_rows.append(excluded_record(subject, task, exercise_folder, None, "Borg row missing"))
                continue
            borg_row = subject_borg.iloc[0]
            trial_length = borg_row["length_of_trial_(sec)"]
            if pd.isna(trial_length):
                excluded_rows.append(excluded_record(subject, task, exercise_folder, None, "Borg trial length missing"))
                continue

            acc_path = subject_dir / "IMU data" / "Upper arm" / "acc_upper_arm.csv"
            gyr_path = subject_dir / "IMU data" / "Upper arm" / "gyr_upper_arm.csv"
            if not acc_path.exists() or not gyr_path.exists():
                missing = acc_path if not acc_path.exists() else gyr_path
                excluded_rows.append(excluded_record(subject, task, exercise_folder, None, "file missing", str(missing)))
                continue
            try:
                acc = load_acc(acc_path)
                gyr = load_gyr(gyr_path)
            except Exception as exc:
                excluded_rows.append(excluded_record(subject, task, exercise_folder, None, "sensor file format error", str(exc)))
                continue

            last_complete_end = int(float(trial_length) // WINDOW_SECONDS) * WINDOW_SECONDS
            for window_end_sec in range(WINDOW_SECONDS, last_complete_end + 1, WINDOW_SECONDS):
                label_col = label_columns.get(window_end_sec)
                if label_col is None or not has_real_rpe(borg_row.get(label_col, pd.NA)):
                    excluded_rows.append(excluded_record(subject, task, exercise_folder, window_end_sec, "Borg RPE missing"))
                    continue
                start_idx, end_idx = window_bounds(window_end_sec)
                if len(acc) < end_idx:
                    excluded_rows.append(excluded_record(subject, task, exercise_folder, window_end_sec, "ACC insufficient length", f"need {end_idx}, got {len(acc)}"))
                    continue
                if len(gyr) < end_idx:
                    excluded_rows.append(excluded_record(subject, task, exercise_folder, window_end_sec, "GYR insufficient length", f"need {end_idx}, got {len(gyr)}"))
                    continue

                window = pd.concat([
                    acc.iloc[start_idx:end_idx][ACC_COLUMNS].reset_index(drop=True),
                    gyr.iloc[start_idx:end_idx][GYR_COLUMNS].reset_index(drop=True),
                ], axis=1)
                if len(window) != WINDOW_SIZE:
                    excluded_rows.append(excluded_record(subject, task, exercise_folder, window_end_sec, "window size error", f"expected {WINDOW_SIZE}, got {len(window)}"))
                    continue
                if has_invalid_sensor_value(window[SENSOR_COLUMNS]):
                    excluded_rows.append(excluded_record(subject, task, exercise_folder, window_end_sec, "invalid sensor value"))
                    continue

                rpe = pd.to_numeric(borg_row[label_col], errors="coerce")
                if pd.isna(rpe):
                    excluded_rows.append(excluded_record(subject, task, exercise_folder, window_end_sec, "Borg RPE invalid"))
                    continue
                row = {
                    "subject": subject,
                    "task": task,
                    "exercise_folder": exercise_folder,
                    "window_end_sec": window_end_sec,
                    "rpe": float(rpe),
                }
                row.update(extract_window_features(window))
                feature_rows.append(row)

    features = pd.DataFrame(feature_rows).reindex(columns=METADATA_COLUMNS + FEATURE_COLUMNS)
    excluded = pd.DataFrame(excluded_rows, columns=["subject", "task", "exercise_folder", "window_end_sec", "reason", "detail"])
    return features, excluded

## 11. 运行原始数据处理并保存特征表

这一格真的从原始 Upper-arm ACC/GYR 和 Borg 表重新构建数据，不读取已有特征表作为捷径。生成文件会写入 V2 项目的 `data_processed`，原始数据仍然只读。

In [ ]:
features_df, excluded_df = build_feature_dataset()
features_df.to_csv(FATIGUE_FEATURES_CSV, index=False, encoding="utf-8-sig")
excluded_df.to_csv(EXCLUDED_WINDOWS_CSV, index=False, encoding="utf-8-sig")

print("fatigue_features.csv:", FATIGUE_FEATURES_CSV)
print("excluded_windows.csv:", EXCLUDED_WINDOWS_CSV)
print("有效窗口:", len(features_df))
print("排除记录:", len(excluded_df))

## 12. 对特征表做严格质量检查

最终数据应为 1310 个窗口、34 名受试者和 56 个特征，并且没有重复窗口、NaN 或 inf。断言可以防止错误数据继续进入模型。

In [ ]:
duplicate_count = int(features_df.duplicated(["subject", "task", "window_end_sec"]).sum())
feature_values = features_df[FEATURE_COLUMNS].to_numpy(dtype=float)
quality_report = {
    "sample_count": len(features_df),
    "subject_count": features_df["subject"].nunique(),
    "task_count": features_df["task"].nunique(),
    "feature_count": len(FEATURE_COLUMNS),
    "nan_count": int(features_df.isna().sum().sum()),
    "inf_count": int(np.isinf(feature_values).sum()),
    "duplicate_windows": duplicate_count,
    "constant_features": int((features_df[FEATURE_COLUMNS].nunique() <= 1).sum()),
    "excluded_records": len(excluded_df),
}

display(pd.DataFrame([quality_report]))
display(features_df.head())
display(features_df["task"].value_counts().sort_index().rename("windows_per_task"))
display(features_df["rpe"].value_counts().sort_index().rename("windows_per_rpe"))
display(excluded_df["reason"].value_counts().rename("excluded_count") if not excluded_df.empty else pd.Series(dtype=int))

assert quality_report["sample_count"] == 1310
assert quality_report["subject_count"] == 34
assert quality_report["feature_count"] == 56
assert quality_report["nan_count"] == 0
assert quality_report["inf_count"] == 0
assert quality_report["duplicate_windows"] == 0

## 13. 明确模型输入 X、回归标签、分类标签和分组变量

模型输入 `X` 只能包含 56 个 IMU 特征。metadata、时间、subject 和标签都不能进入模型。

- `y_reg = rpe`：连续值回归目标。
- `y_cls = (rpe >= 17)`：实验性高疲劳二分类标签。
- `groups = subject`：GroupKFold 的分组依据。

`RPE>=17` 是标签定义；后面的 `probability threshold` 是把分类概率转为 0/1 的决策门槛，两者不是一回事。

In [ ]:
df = features_df.copy()
forbidden_columns = {
    "subject", "task", "exercise_folder", "window_end_sec", "rpe", "high_fatigue"
}

X = df[FEATURE_COLUMNS].copy()
y_reg = df["rpe"].astype(float).copy()         #         # y_reg = df["rpe"]  是取一列的意思。
y_cls = (df["rpe"] >= 17).astype(int)
groups = df["subject"].copy()
metadata = df[["subject", "task", "exercise_folder", "window_end_sec"]].copy()

print("X.shape:", X.shape)
print("特征数量:", X.shape[1])
print("subject 数量:", groups.nunique())
print("high_fatigue=0:", int((y_cls == 0).sum()))
print("high_fatigue=1:", int((y_cls == 1).sum()))
print("类别占比:")
display(y_cls.value_counts(normalize=True).sort_index().rename("proportion"))

assert forbidden_columns.isdisjoint(X.columns)
assert X.shape == (1310, 56)
assert groups.nunique() == 34
assert len(X) == len(y_reg) == len(y_cls) == len(groups)

## 14. 先手动演示外层 GroupKFold 第一折

普通随机 8:2 会把同一受试者的不同窗口同时放进训练和测试，造成 subject 泄漏。这里按 subject 分组，保证验证受试者在训练中完全没出现。

外层 5 折的职责只有一个：评估模型对未见受试者的泛化能力。

In [ ]:
outer_cv_demo = GroupKFold(n_splits=N_SPLITS_OUTER)
first_train_idx, first_val_idx = next(outer_cv_demo.split(X, y_reg, groups=groups))

first_train_subjects = sorted(set(groups.iloc[first_train_idx]))
first_val_subjects = sorted(set(groups.iloc[first_val_idx]))
first_intersection = set(first_train_subjects) & set(first_val_subjects)

print("train_idx 前 20 个:", first_train_idx[:20])
print("val_idx 前 20 个:", first_val_idx[:20])
print("训练样本数:", len(first_train_idx))
print("验证样本数:", len(first_val_idx))
print("训练 subject:", first_train_subjects)
print("验证 subject:", first_val_subjects)
print("交集:", first_intersection)

assert len(first_intersection) == 0

## 15. 再手动演示内层 GroupKFold 第一折

Nested GroupKFold 中，调参只能发生在 `outer_train` 内部。这里把第一折的 outer train 再分成 4 个 inner folds，确认内层训练和内层验证也没有 subject 交集。

因此本项目不是“回归随机调参、分类 Nested GroupKFold”。正确关系是：

- 回归和分类都使用外层 5 折 + 内层 4 折的 Nested GroupKFold。
- GridSearchCV/RandomizedSearchCV 只是内层尝试参数组合的搜索方法。

In [ ]:
X_outer_train_demo = X.iloc[first_train_idx]
y_outer_train_demo = y_reg.iloc[first_train_idx]
groups_outer_train_demo = groups.iloc[first_train_idx]

inner_cv_demo = GroupKFold(n_splits=N_SPLITS_INNER)
inner_train_idx, inner_val_idx = next(inner_cv_demo.split(
    X_outer_train_demo,
    y_outer_train_demo,
    groups=groups_outer_train_demo,
))

inner_train_subjects = set(groups_outer_train_demo.iloc[inner_train_idx])
inner_val_subjects = set(groups_outer_train_demo.iloc[inner_val_idx])

print("内层训练 subject 数:", len(inner_train_subjects))
print("内层验证 subject 数:", len(inner_val_subjects))
print("内层 subject 交集:", inner_train_subjects & inner_val_subjects)
assert not (inner_train_subjects & inner_val_subjects)

## 16. 定义统一的回归和分类指标函数

回归使用 MAE、RMSE、R²。模型选择主指标是 MAE，越小越好。

分类同时报告 Accuracy、Balanced Accuracy、Precision、Recall、F1、F2、ROC-AUC、Specificity、Miss Rate、FPR 和混淆矩阵四项。模型选择主指标是 ROC-AUC；预警阈值选择主指标才是 F2。

In [ ]:
def regression_metrics(y_true, y_pred) -> dict[str, float]:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }


def classification_metrics(y_true, pred, prob) -> dict[str, float]:
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    recall = recall_score(y_true, pred, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) else 0.0
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "f2": float(fbeta_score(y_true, pred, beta=2, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, prob)),
        "specificity": float(specificity),
        "miss_rate": float(1 - recall),
        "false_positive_rate": float(false_positive_rate),
        "predicted_positive_rate": float(np.mean(pred)),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }

## 17. 定义可复用的外层分组切分函数

这个函数每次都创建相同的 5 折 subject-level GroupKFold，并在每一折执行泄漏断言。

同时记录每折的训练与验证受试者，最终写入 CSV，保证每条 OOF 结果可以追溯。

In [ ]:
def outer_splits(X_data, y_data, group_data):
    splitter = GroupKFold(n_splits=N_SPLITS_OUTER)
    for fold, (train_idx, val_idx) in enumerate(
        splitter.split(X_data, y_data, groups=group_data), start=1
    ):
        train_subjects = sorted(set(group_data.iloc[train_idx]))
        val_subjects = sorted(set(group_data.iloc[val_idx]))
        intersection = set(train_subjects) & set(val_subjects)
        assert not intersection, f"Subject leakage in outer fold {fold}: {intersection}"
        yield fold, train_idx, val_idx, train_subjects, val_subjects


fold_subject_rows = []
for fold, train_idx, val_idx, train_subjects, val_subjects in outer_splits(X, y_reg, groups):
    fold_subject_rows.append({
        "outer_fold": fold,
        "n_train_samples": len(train_idx),
        "n_val_samples": len(val_idx),
        "n_train_subjects": len(train_subjects),
        "n_val_subjects": len(val_subjects),
        "train_subjects": "|".join(train_subjects),
        "val_subjects": "|".join(val_subjects),
    })

outer_fold_subjects = pd.DataFrame(fold_subject_rows)
outer_fold_subjects.to_csv(RESULTS_DIR / "outer_fold_subjects.csv", index=False, encoding="utf-8-sig")
display(outer_fold_subjects)

## 18. 定义回归 baseline 模型

baseline 先回答“不同模型在相同 5 折 GroupKFold 下大致表现如何”，此时不调参。

- Dummy：只预测训练集 RPE 平均值。
- Ridge：线性模型，标准化放进 Pipeline，避免 CV 前全局标准化泄漏。
- Random Forest：保留旧 Phase 3 的 300 棵树设置。
- XGBoost：默认稳健基线。
- SVR：RBF 核，标准化放进 Pipeline。

In [ ]:
def regression_baseline_models() -> dict[str, Any]:
    return {
        "Dummy": DummyRegressor(strategy="mean"),
        "Ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "RandomForest": RandomForestRegressor(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "XGBoost": XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "SVR": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVR(kernel="rbf")),
        ]),
    }


for name, model in regression_baseline_models().items():
    print(name, "->", model)

## 19. 运行回归 baseline 并生成 OOF prediction

每个样本只在所属外层验证折中获得一次预测，所以每个模型必须恰好有 1310 条 OOF 记录。`subject/task/window_end_sec` 一直保留用于追踪，但不进入 X。

In [ ]:
reg_baseline_fold_rows = []
reg_baseline_pred_rows = []

for fold, train_idx, val_idx, train_subjects, val_subjects in outer_splits(X, y_reg, groups):
    for model_name, model in regression_baseline_models().items():
        estimator = clone(model)
        estimator.fit(X.iloc[train_idx], y_reg.iloc[train_idx])
        pred = estimator.predict(X.iloc[val_idx])
        metrics = regression_metrics(y_reg.iloc[val_idx], pred)
        reg_baseline_fold_rows.append({
            "model": model_name,
            "fold": fold,
            "n_train_samples": len(train_idx),
            "n_val_samples": len(val_idx),
            "n_train_subjects": len(train_subjects),
            "n_val_subjects": len(val_subjects),
            **metrics,
        })
        fold_pred = metadata.iloc[val_idx].copy()
        fold_pred["true_rpe"] = y_reg.iloc[val_idx].to_numpy()
        fold_pred["pred_rpe"] = pred
        fold_pred["absolute_error"] = np.abs(fold_pred["pred_rpe"] - fold_pred["true_rpe"])
        fold_pred["signed_error"] = fold_pred["pred_rpe"] - fold_pred["true_rpe"]
        fold_pred["fold"] = fold
        fold_pred["model"] = model_name
        reg_baseline_pred_rows.append(fold_pred)

reg_baseline_fold_metrics = pd.DataFrame(reg_baseline_fold_rows)
reg_baseline_oof = pd.concat(reg_baseline_pred_rows, ignore_index=True)

reg_baseline_summary_rows = []
for model_name, model_fold_df in reg_baseline_fold_metrics.groupby("model"):
    model_oof = reg_baseline_oof[reg_baseline_oof["model"] == model_name]
    overall = regression_metrics(model_oof["true_rpe"], model_oof["pred_rpe"])
    reg_baseline_summary_rows.append({
        "model": model_name,
        "overall_mae": overall["mae"],
        "overall_rmse": overall["rmse"],
        "overall_r2": overall["r2"],
        "fold_mae_mean": model_fold_df["mae"].mean(),
        "fold_mae_std": model_fold_df["mae"].std(ddof=1),
    })

regression_baseline_comparison = pd.DataFrame(reg_baseline_summary_rows).sort_values("overall_mae")
regression_baseline_comparison.to_csv(RESULTS_DIR / "regression_baseline_comparison.csv", index=False, encoding="utf-8-sig")
reg_baseline_fold_metrics.to_csv(RESULTS_DIR / "regression_baseline_fold_metrics.csv", index=False, encoding="utf-8-sig")
reg_baseline_oof.to_csv(RESULTS_DIR / "regression_baseline_oof_predictions.csv", index=False, encoding="utf-8-sig")

display(regression_baseline_comparison)
display(reg_baseline_oof.groupby("model").size().rename("oof_rows"))
assert (reg_baseline_oof.groupby("model").size() == len(df)).all()

## 20. 定义分类 baseline 模型

分类标签固定为 `RPE>=17`。baseline 比较 Logistic Regression、Random Forest Classifier 与 XGBoost Classifier。

默认类别判断均使用 probability threshold=0.5。这个 0.5 只是概率决策阈值，不会改变真实标签的 `RPE>=17` 定义。

In [ ]:
def classification_baseline_models() -> dict[str, Any]:
    return {
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
        ]),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
        "XGBoost": XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }


for name, model in classification_baseline_models().items():
    print(name, "->", model)

## 21. 运行分类 baseline 并生成 OOF probability

这里必须使用 `predict_proba(... )[:, 1]` 得到高疲劳概率，再用 0.5 转成预测类别。不能把 0/1 预测结果冒充概率。

模型比较以 ROC-AUC 为主，同时完整报告 Recall、Precision、F1、Balanced Accuracy 和混淆矩阵。

In [ ]:
cls_baseline_fold_rows = []
cls_baseline_pred_rows = []

for fold, train_idx, val_idx, train_subjects, val_subjects in outer_splits(X, y_cls, groups):
    for model_name, model in classification_baseline_models().items():
        estimator = clone(model)
        estimator.fit(X.iloc[train_idx], y_cls.iloc[train_idx])
        probability = estimator.predict_proba(X.iloc[val_idx])[:, 1]
        pred = (probability >= 0.5).astype(int)
        metrics = classification_metrics(y_cls.iloc[val_idx], pred, probability)
        cls_baseline_fold_rows.append({
            "model": model_name,
            "fold": fold,
            "n_train_samples": len(train_idx),
            "n_val_samples": len(val_idx),
            "n_train_subjects": len(train_subjects),
            "n_val_subjects": len(val_subjects),
            **metrics,
        })
        fold_pred = metadata.iloc[val_idx].copy()
        fold_pred["rpe"] = y_reg.iloc[val_idx].to_numpy()
        fold_pred["true_high_fatigue"] = y_cls.iloc[val_idx].to_numpy()
        fold_pred["pred_high_fatigue_probability"] = probability
        fold_pred["pred_high_fatigue"] = pred
        fold_pred["fold"] = fold
        fold_pred["model"] = model_name
        cls_baseline_pred_rows.append(fold_pred)

cls_baseline_fold_metrics = pd.DataFrame(cls_baseline_fold_rows)
cls_baseline_oof = pd.concat(cls_baseline_pred_rows, ignore_index=True)

cls_baseline_summary_rows = []
for model_name, model_fold_df in cls_baseline_fold_metrics.groupby("model"):
    model_oof = cls_baseline_oof[cls_baseline_oof["model"] == model_name]
    overall = classification_metrics(
        model_oof["true_high_fatigue"],
        model_oof["pred_high_fatigue"].to_numpy(),
        model_oof["pred_high_fatigue_probability"].to_numpy(),
    )
    cls_baseline_summary_rows.append({
        "model": model_name,
        "oof_roc_auc": overall["roc_auc"],
        "precision_at_0_5": overall["precision"],
        "recall_at_0_5": overall["recall"],
        "f1_at_0_5": overall["f1"],
        "accuracy_at_0_5": overall["accuracy"],
        "balanced_accuracy_at_0_5": overall["balanced_accuracy"],
        "specificity_at_0_5": overall["specificity"],
        "TN": overall["TN"], "FP": overall["FP"], "FN": overall["FN"], "TP": overall["TP"],
        "fold_auc_mean": model_fold_df["roc_auc"].mean(),
        "fold_auc_std": model_fold_df["roc_auc"].std(ddof=1),
    })

classification_baseline_comparison = pd.DataFrame(cls_baseline_summary_rows).sort_values("oof_roc_auc", ascending=False)
classification_baseline_comparison.to_csv(RESULTS_DIR / "classification_baseline_comparison.csv", index=False, encoding="utf-8-sig")
cls_baseline_fold_metrics.to_csv(RESULTS_DIR / "classification_baseline_fold_metrics.csv", index=False, encoding="utf-8-sig")
cls_baseline_oof.to_csv(RESULTS_DIR / "classification_baseline_oof_predictions.csv", index=False, encoding="utf-8-sig")

display(classification_baseline_comparison)
display(cls_baseline_oof.groupby("model").size().rename("oof_rows"))
assert (cls_baseline_oof.groupby("model").size() == len(df)).all()

## 22. 定义回归正式调参空间

回归正式实验对所有模型使用相同的 Nested GroupKFold 框架，但模型的参数搜索方式不同：

- Dummy：没有参数，不搜索。
- Ridge：候选很少，使用完整 GridSearchCV。
- Random Forest、XGBoost、SVR：组合较多，使用 20 次 RandomizedSearchCV。

无论 Grid 还是 Randomized，它们都只在当前 outer train 的内层 4 折 GroupKFold 中运行，scoring 为 `neg_mean_absolute_error`。

In [ ]:
def regression_search_spaces() -> dict[str, tuple[Any, dict[str, Any], str]]:
    return {
        "Dummy": (
            DummyRegressor(strategy="mean"),
            {},
            "none",
        ),
        "Ridge": (
            Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
            {"model__alpha": [0.01, 0.1, 1, 10, 100]},
            "grid",
        ),
        "RandomForest": (
            RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
            {
                "n_estimators": [200, 300, 500, 800],
                "max_depth": [None, 5, 10, 15, 20],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4, 8],
                "max_features": ["sqrt", 0.5, 1.0],
            },
            "random",
        ),
        "XGBoost": (
            XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1),
            {
                "n_estimators": [100, 200, 300, 500],
                "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
                "max_depth": [2, 3, 4, 5, 6],
                "subsample": [0.7, 0.85, 1.0],
                "colsample_bytree": [0.7, 0.85, 1.0],
                "min_child_weight": [1, 3, 5],
                "reg_alpha": [0, 0.01, 0.1],
                "reg_lambda": [1, 5, 10],
            },
            "random",
        ),
        "SVR": (
            Pipeline([("scaler", StandardScaler()), ("model", SVR(kernel="rbf"))]),
            {
                "model__C": [0.1, 1, 10, 100],
                "model__gamma": ["scale", 0.001, 0.01, 0.1],
                "model__epsilon": [0.1, 0.25, 0.5, 1.0],
            },
            "random",
        ),
    }


for model_name, (_, params, mode) in regression_search_spaces().items():
    print(model_name, "| search=", mode, "| parameters=", params)

## 23. 定义分类正式调参空间

分类同样使用外层 5 折、内层 4 折：

- Logistic Regression：10 个组合，使用 GridSearchCV。
- Random Forest、XGBoost：使用 20 次 RandomizedSearchCV。
- 内层 scoring 固定为 ROC-AUC，先优化概率排序能力，不在这一步混入 warning threshold。

In [ ]:
def classification_search_spaces() -> dict[str, tuple[Any, dict[str, Any], str]]:
    return {
        "LogisticRegression": (
            Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
            ]),
            {
                "model__C": [0.01, 0.1, 1, 10, 100],
                "model__class_weight": [None, "balanced"],
            },
            "grid",
        ),
        "RandomForest": (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
            {
                "n_estimators": [200, 300, 500, 800],
                "max_depth": [None, 5, 10, 15, 20],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4, 8],
                "max_features": ["sqrt", 0.5, 1.0],
                "class_weight": [None, "balanced", "balanced_subsample"],
            },
            "random",
        ),
        "XGBoost": (
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            {
                "n_estimators": [100, 200, 300, 500],
                "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
                "max_depth": [2, 3, 4, 5, 6],
                "subsample": [0.7, 0.85, 1.0],
                "colsample_bytree": [0.7, 0.85, 1.0],
                "min_child_weight": [1, 3, 5],
                "reg_alpha": [0, 0.01, 0.1],
                "reg_lambda": [1, 5, 10],
            },
            "random",
        ),
    }


for model_name, (_, params, mode) in classification_search_spaces().items():
    print(model_name, "| search=", mode, "| parameters=", params)

## 24. 定义内层参数搜索器

这个辅助函数只决定“候选参数怎样被搜索”，不决定数据怎样切分。`inner_cv` 永远是调用处传入的 4 折 GroupKFold。

StandardScaler 被放在 Ridge、Logistic 和 SVR 的 Pipeline 中，因此每个 inner train 会单独拟合 scaler，不会看到 inner validation 或 outer validation。

In [ ]:
def make_parameter_search(estimator, params, mode, scoring, inner_cv):
    if mode == "grid":
        return GridSearchCV(
            estimator=estimator,
            param_grid=params,
            scoring=scoring,
            cv=inner_cv,
            n_jobs=-1,
        )
    if mode == "random":
        return RandomizedSearchCV(
            estimator=estimator,
            param_distributions=params,
            n_iter=N_ITER_RANDOM_SEARCH,
            scoring=scoring,
            cv=inner_cv,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    raise ValueError(f"未知搜索模式: {mode}")

## 25. 定义回归 Nested GroupKFold 的完整执行函数

对每个模型、每个 outer fold：

1. outer train 内建立 4 折 GroupKFold。
2. 用 inner folds 选择 MAE 最优参数。
3. 用最佳参数在完整 outer train 拟合。
4. 只对未见过的 outer validation subjects 预测。
5. 保存外层指标、参数、subject 列表和 OOF 记录。

最终模型比较使用拼接全部外层预测后的 overall OOF MAE。

In [ ]:
def run_nested_regression(X_data, y_data, group_data, metadata_data):
    fold_rows, param_rows, prediction_rows = [], [], []

    for model_name, (base_estimator, params, mode) in regression_search_spaces().items():
        print(f"[Nested regression] {model_name}")
        for fold, train_idx, val_idx, train_subjects, val_subjects in outer_splits(
            X_data, y_data, group_data
        ):
            if mode == "none":
                estimator = clone(base_estimator)
                best_params = {}
                inner_best_score = np.nan
            else:
                inner_cv = GroupKFold(n_splits=N_SPLITS_INNER)
                search = make_parameter_search(
                    clone(base_estimator),
                    params,
                    mode,
                    scoring="neg_mean_absolute_error",
                    inner_cv=inner_cv,
                )
                search.fit(
                    X_data.iloc[train_idx],
                    y_data.iloc[train_idx],
                    groups=group_data.iloc[train_idx],
                )
                estimator = clone(search.best_estimator_)
                best_params = search.best_params_
                inner_best_score = float(search.best_score_)

            estimator.fit(X_data.iloc[train_idx], y_data.iloc[train_idx])
            pred = estimator.predict(X_data.iloc[val_idx])
            metrics = regression_metrics(y_data.iloc[val_idx], pred)
            fold_rows.append({
                "model": model_name,
                "outer_fold": fold,
                "n_train_samples": len(train_idx),
                "n_val_samples": len(val_idx),
                "n_train_subjects": len(train_subjects),
                "n_val_subjects": len(val_subjects),
                **metrics,
            })
            param_rows.append({
                "model": model_name,
                "outer_fold": fold,
                "search_mode": mode,
                "inner_scoring": "neg_mean_absolute_error",
                "inner_best_score": inner_best_score,
                "best_params": json.dumps(best_params, ensure_ascii=False),
                "train_subjects": "|".join(train_subjects),
                "val_subjects": "|".join(val_subjects),
            })
            fold_pred = metadata_data.iloc[val_idx].copy()
            fold_pred["true_rpe"] = y_data.iloc[val_idx].to_numpy()
            fold_pred["pred_rpe"] = pred
            fold_pred["absolute_error"] = np.abs(fold_pred["pred_rpe"] - fold_pred["true_rpe"])
            fold_pred["signed_error"] = fold_pred["pred_rpe"] - fold_pred["true_rpe"]
            fold_pred["outer_fold"] = fold
            fold_pred["model"] = model_name
            prediction_rows.append(fold_pred)

    fold_metrics = pd.DataFrame(fold_rows)
    best_params_df = pd.DataFrame(param_rows)
    outer_predictions = pd.concat(prediction_rows, ignore_index=True)

    summary_rows = []
    for model_name, model_fold_df in fold_metrics.groupby("model"):
        model_oof = outer_predictions[outer_predictions["model"] == model_name]
        overall = regression_metrics(model_oof["true_rpe"], model_oof["pred_rpe"])
        summary_rows.append({
            "model": model_name,
            "overall_mae": overall["mae"],
            "overall_rmse": overall["rmse"],
            "overall_r2": overall["r2"],
            "fold_mae_mean": model_fold_df["mae"].mean(),
            "fold_mae_std": model_fold_df["mae"].std(ddof=1),
            "fold_rmse_mean": model_fold_df["rmse"].mean(),
            "fold_rmse_std": model_fold_df["rmse"].std(ddof=1),
            "fold_r2_mean": model_fold_df["r2"].mean(),
            "fold_r2_std": model_fold_df["r2"].std(ddof=1),
        })
    summary = pd.DataFrame(summary_rows).sort_values("overall_mae")
    return summary, fold_metrics, best_params_df, outer_predictions

## 26. 运行回归 Nested GroupKFold 并选择最佳模型

这一步计算量较大，因为每个模型都要经历 5 个 outer folds，且每个 outer train 内再次进行 4 折搜索。

最佳回归模型只按真实外层 OOF MAE 选择。不会因为某次训练集结果好或某一个 fold 好就选择它。

In [ ]:
(
    regression_nested_metrics,
    regression_nested_fold_metrics,
    regression_best_params,
    regression_outer_predictions,
) = run_nested_regression(X, y_reg, groups, metadata)

regression_nested_metrics.to_csv(RESULTS_DIR / "regression_nested_metrics.csv", index=False, encoding="utf-8-sig")
regression_nested_fold_metrics.to_csv(RESULTS_DIR / "regression_nested_fold_metrics.csv", index=False, encoding="utf-8-sig")
regression_best_params.to_csv(RESULTS_DIR / "regression_best_params.csv", index=False, encoding="utf-8-sig")
regression_outer_predictions.to_csv(RESULTS_DIR / "regression_outer_predictions.csv", index=False, encoding="utf-8-sig")

best_regression_model = regression_nested_metrics.iloc[0]["model"]
best_regression_row = regression_nested_metrics.iloc[0]

display(regression_nested_metrics)
print("Best Regression Model:", best_regression_model)
print("MAE:", best_regression_row["overall_mae"])
print("RMSE:", best_regression_row["overall_rmse"])
print("R²:", best_regression_row["overall_r2"])
display(regression_best_params[regression_best_params["model"] == best_regression_model])

assert (regression_outer_predictions.groupby("model").size() == len(df)).all()

## 27. 定义分类 Nested GroupKFold 的完整执行函数

流程与回归相同，但内层选择参数的评分是 ROC-AUC，outer validation 输出的是 `predict_proba`。

外层 OOF probability 用于公平比较 Logistic、RF 与 XGBoost；默认 0.5 指标只是比较表的一部分，尚未进行预警阈值优化。

In [ ]:
def run_nested_classification(X_data, y_data, group_data, metadata_data, rpe_data):
    fold_rows, param_rows, prediction_rows = [], [], []

    for model_name, (base_estimator, params, mode) in classification_search_spaces().items():
        print(f"[Nested classification] {model_name}")
        for fold, train_idx, val_idx, train_subjects, val_subjects in outer_splits(
            X_data, y_data, group_data
        ):
            inner_cv = GroupKFold(n_splits=N_SPLITS_INNER)
            search = make_parameter_search(
                clone(base_estimator),
                params,
                mode,
                scoring="roc_auc",
                inner_cv=inner_cv,
            )
            search.fit(
                X_data.iloc[train_idx],
                y_data.iloc[train_idx],
                groups=group_data.iloc[train_idx],
            )
            estimator = clone(search.best_estimator_)
            estimator.fit(X_data.iloc[train_idx], y_data.iloc[train_idx])
            probability = estimator.predict_proba(X_data.iloc[val_idx])[:, 1]
            pred = (probability >= 0.5).astype(int)
            metrics = classification_metrics(y_data.iloc[val_idx], pred, probability)

            fold_rows.append({
                "model": model_name,
                "outer_fold": fold,
                "n_train_samples": len(train_idx),
                "n_val_samples": len(val_idx),
                "n_train_subjects": len(train_subjects),
                "n_val_subjects": len(val_subjects),
                **metrics,
            })
            param_rows.append({
                "model": model_name,
                "outer_fold": fold,
                "search_mode": mode,
                "inner_scoring": "roc_auc",
                "inner_best_score": float(search.best_score_),
                "best_params": json.dumps(search.best_params_, ensure_ascii=False),
                "train_subjects": "|".join(train_subjects),
                "val_subjects": "|".join(val_subjects),
            })
            fold_pred = metadata_data.iloc[val_idx].copy()
            fold_pred["rpe"] = rpe_data.iloc[val_idx].to_numpy()
            fold_pred["true_high_fatigue"] = y_data.iloc[val_idx].to_numpy()
            fold_pred["pred_high_fatigue_probability"] = probability
            fold_pred["pred_high_fatigue"] = pred
            fold_pred["outer_fold"] = fold
            fold_pred["model"] = model_name
            prediction_rows.append(fold_pred)

    fold_metrics = pd.DataFrame(fold_rows)
    best_params_df = pd.DataFrame(param_rows)
    outer_probabilities = pd.concat(prediction_rows, ignore_index=True)

    summary_rows = []
    for model_name, model_fold_df in fold_metrics.groupby("model"):
        model_oof = outer_probabilities[outer_probabilities["model"] == model_name]
        overall = classification_metrics(
            model_oof["true_high_fatigue"],
            model_oof["pred_high_fatigue"].to_numpy(),
            model_oof["pred_high_fatigue_probability"].to_numpy(),
        )
        summary_rows.append({
            "model": model_name,
            "oof_roc_auc": overall["roc_auc"],
            "precision_at_0_5": overall["precision"],
            "recall_at_0_5": overall["recall"],
            "f1_at_0_5": overall["f1"],
            "accuracy_at_0_5": overall["accuracy"],
            "balanced_accuracy_at_0_5": overall["balanced_accuracy"],
            "specificity_at_0_5": overall["specificity"],
            "TN": overall["TN"], "FP": overall["FP"], "FN": overall["FN"], "TP": overall["TP"],
            "fold_auc_mean": model_fold_df["roc_auc"].mean(),
            "fold_auc_std": model_fold_df["roc_auc"].std(ddof=1),
        })
    summary = pd.DataFrame(summary_rows).sort_values("oof_roc_auc", ascending=False)
    return summary, fold_metrics, best_params_df, outer_probabilities

## 28. 运行分类 Nested GroupKFold 并选择最佳模型

最佳分类模型按外层 OOF ROC-AUC 选择，因为 AUC 衡量概率对正负样本的排序能力，不依赖某一个固定阈值。Recall、Precision、F1 和 Balanced Accuracy 同时报告，但不单独用 Accuracy 选模型。

In [ ]:
(
    classification_nested_metrics,
    classification_nested_fold_metrics,
    classification_best_params,
    classification_outer_probabilities,
) = run_nested_classification(X, y_cls, groups, metadata, y_reg)

classification_nested_metrics.to_csv(RESULTS_DIR / "classification_nested_metrics.csv", index=False, encoding="utf-8-sig")
classification_nested_fold_metrics.to_csv(RESULTS_DIR / "classification_nested_fold_metrics.csv", index=False, encoding="utf-8-sig")
classification_best_params.to_csv(RESULTS_DIR / "classification_best_params.csv", index=False, encoding="utf-8-sig")
classification_outer_probabilities.to_csv(RESULTS_DIR / "classification_outer_probabilities.csv", index=False, encoding="utf-8-sig")

best_classification_model = classification_nested_metrics.iloc[0]["model"]
best_classification_row = classification_nested_metrics.iloc[0]

display(classification_nested_metrics)
print("Best Classification Model:", best_classification_model)
print("ROC-AUC:", best_classification_row["oof_roc_auc"])
print("Recall@0.5:", best_classification_row["recall_at_0_5"])
print("F1@0.5:", best_classification_row["f1_at_0_5"])
display(classification_best_params[classification_best_params["model"] == best_classification_model])

assert (classification_outer_probabilities.groupby("model").size() == len(df)).all()

## 29. 定义“已选参数的分类器”重建函数

分类 Nested GroupKFold 已经为最佳分类模型的每个 outer fold 保存了一组 `best_params`。阈值阶段必须使用对应 fold 的参数，不能重新看 outer validation 标签选模型参数。

In [ ]:
def classification_estimator_from_params(model_name: str, params: dict[str, Any]):
    base_estimator, _, _ = classification_search_spaces()[model_name]
    estimator = clone(base_estimator)
    estimator.set_params(**params)
    return estimator


best_cls_param_preview = classification_best_params[
    classification_best_params["model"] == best_classification_model
].copy()
display(best_cls_param_preview[["model", "outer_fold", "best_params", "inner_best_score"]])

## 30. 定义 F2 阈值曲线和选择规则

模型参数和概率阈值是两个不同层次：

- 超参数决定模型如何拟合与输出概率。
- threshold 决定概率达到多少时发出实验性预警。

候选阈值从 0.05 到 0.80，步长 0.01。选择 F2 最大的阈值；F2 比 F1 更重视 Recall。阈值降低通常减少 FN，但会增加 FP。

In [ ]:
THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.01), 2)


def build_threshold_curve(y_true, probability, thresholds=THRESHOLDS) -> pd.DataFrame:
    rows = []
    for threshold in thresholds:
        pred = (probability >= threshold).astype(int)
        rows.append({
            "threshold": float(threshold),
            **classification_metrics(y_true, pred, probability),
        })
    return pd.DataFrame(rows)


def select_f2_threshold(curve: pd.DataFrame) -> pd.Series:
    return curve.sort_values(
        ["f2", "recall", "threshold"],
        ascending=[False, False, False],
    ).iloc[0]


print("候选阈值数量:", len(THRESHOLDS))
print("范围:", THRESHOLDS[0], "to", THRESHOLDS[-1])

## 31. 在每个 outer train 内部选择预警阈值

这一格是避免阈值数据泄漏的关键。对每个 outer fold：

1. 读取该折已经由 Nested CV 选好的分类模型参数。
2. 只在 outer train 受试者内部再做 4 折 GroupKFold。
3. 得到 outer train 的 inner OOF probability。
4. 只用这些训练侧概率和标签选择 F2 阈值。
5. 最后把这个阈值应用到从未参与选择的 outer validation probability。

outer validation 的真实标签绝不参与阈值选择。

In [ ]:
threshold_prediction_rows = []
threshold_selection_rows = []
threshold_curve_rows = []

for fold, train_idx, val_idx, train_subjects, val_subjects in outer_splits(X, y_cls, groups):
    params_text = classification_best_params[
        (classification_best_params["model"] == best_classification_model)
        & (classification_best_params["outer_fold"] == fold)
    ].iloc[0]["best_params"]
    best_params_for_fold = json.loads(params_text)

    X_outer_train = X.iloc[train_idx].reset_index(drop=True)
    y_outer_train = y_cls.iloc[train_idx].reset_index(drop=True)
    groups_outer_train = groups.iloc[train_idx].reset_index(drop=True)
    inner_oof_probability = np.full(len(X_outer_train), np.nan)

    inner_splitter = GroupKFold(n_splits=N_SPLITS_INNER)
    for inner_train_idx, inner_val_idx in inner_splitter.split(
        X_outer_train, y_outer_train, groups=groups_outer_train
    ):
        inner_train_subjects = set(groups_outer_train.iloc[inner_train_idx])
        inner_val_subjects = set(groups_outer_train.iloc[inner_val_idx])
        assert not (inner_train_subjects & inner_val_subjects)

        estimator = classification_estimator_from_params(
            best_classification_model, best_params_for_fold
        )
        estimator.fit(
            X_outer_train.iloc[inner_train_idx],
            y_outer_train.iloc[inner_train_idx],
        )
        inner_oof_probability[inner_val_idx] = estimator.predict_proba(
            X_outer_train.iloc[inner_val_idx]
        )[:, 1]

    assert not np.isnan(inner_oof_probability).any()
    curve = build_threshold_curve(y_outer_train, inner_oof_probability)
    curve["outer_fold"] = fold
    threshold_curve_rows.append(curve)
    selected = select_f2_threshold(curve)
    selected_threshold = float(selected["threshold"])

    threshold_selection_rows.append({
        "outer_fold": fold,
        "selected_threshold": selected_threshold,
        "inner_precision": selected["precision"],
        "inner_recall": selected["recall"],
        "inner_f1": selected["f1"],
        "inner_f2": selected["f2"],
        "inner_balanced_accuracy": selected["balanced_accuracy"],
        "inner_miss_rate": selected["miss_rate"],
        "inner_false_positive_rate": selected["false_positive_rate"],
        "train_subjects": "|".join(train_subjects),
        "val_subjects": "|".join(val_subjects),
    })

    final_fold_estimator = classification_estimator_from_params(
        best_classification_model, best_params_for_fold
    )
    final_fold_estimator.fit(X.iloc[train_idx], y_cls.iloc[train_idx])
    outer_probability = final_fold_estimator.predict_proba(X.iloc[val_idx])[:, 1]

    fold_prediction = metadata.iloc[val_idx].copy()
    fold_prediction["rpe"] = y_reg.iloc[val_idx].to_numpy()
    fold_prediction["true_high_fatigue"] = y_cls.iloc[val_idx].to_numpy()
    fold_prediction["pred_high_fatigue_probability"] = outer_probability
    fold_prediction["pred_default_05"] = (outer_probability >= 0.5).astype(int)
    fold_prediction["pred_optimized"] = (outer_probability >= selected_threshold).astype(int)
    fold_prediction["selected_threshold"] = selected_threshold
    fold_prediction["outer_fold"] = fold
    fold_prediction["classification_model"] = best_classification_model
    threshold_prediction_rows.append(fold_prediction)

threshold_outer_results = pd.concat(threshold_prediction_rows, ignore_index=True)
threshold_selection_final = pd.DataFrame(threshold_selection_rows)
threshold_curve_final = pd.concat(threshold_curve_rows, ignore_index=True)

display(threshold_selection_final)
assert len(threshold_outer_results) == len(df)
assert threshold_outer_results.duplicated(["subject", "task", "window_end_sec"]).sum() == 0

## 32. 比较默认阈值与 F2 优化阈值

最终必须完整报告收益和代价：Recall 提高多少、FN 减少多少，同时 FP/FPR 增加多少、Precision 如何变化。

同一批 probability 只改变 threshold，所以 ROC-AUC 不会变化。

In [ ]:
threshold_summary_rows = []
for threshold_type, prediction_column in [
    ("default_0.5", "pred_default_05"),
    ("optimized", "pred_optimized"),
]:
    threshold_summary_rows.append({
        "threshold_type": threshold_type,
        **classification_metrics(
            threshold_outer_results["true_high_fatigue"],
            threshold_outer_results[prediction_column].to_numpy(),
            threshold_outer_results["pred_high_fatigue_probability"].to_numpy(),
        ),
    })

threshold_summary = pd.DataFrame(threshold_summary_rows)
threshold_outer_results.to_csv(RESULTS_DIR / "threshold_outer_results.csv", index=False, encoding="utf-8-sig")
threshold_selection_final.to_csv(RESULTS_DIR / "threshold_selection_final.csv", index=False, encoding="utf-8-sig")
threshold_curve_final.to_csv(RESULTS_DIR / "threshold_curve_final.csv", index=False, encoding="utf-8-sig")
threshold_summary.to_csv(RESULTS_DIR / "threshold_summary.csv", index=False, encoding="utf-8-sig")

display(threshold_summary)
default_metrics = threshold_summary[threshold_summary["threshold_type"] == "default_0.5"].iloc[0]
optimized_metrics = threshold_summary[threshold_summary["threshold_type"] == "optimized"].iloc[0]

print("Recall change:", optimized_metrics["recall"] - default_metrics["recall"])
print("Precision change:", optimized_metrics["precision"] - default_metrics["precision"])
print("FN change:", int(optimized_metrics["FN"] - default_metrics["FN"]))
print("FP change:", int(optimized_metrics["FP"] - default_metrics["FP"]))
print("FPR change:", optimized_metrics["false_positive_rate"] - default_metrics["false_positive_rate"])
print("AUC unchanged:", default_metrics["roc_auc"], optimized_metrics["roc_auc"])

## 33. 生成三张最终实验主表

为避免回归、分类和阈值结果混在一起，这里分别整理：

1. 回归：Model、MAE、RMSE、R²。
2. 分类：Model、AUC、Precision@0.5、Recall@0.5、F1@0.5、Balanced Accuracy@0.5。
3. 阈值：默认 0.5 与 F2 optimized 的完整权衡。

In [ ]:
final_regression_table = regression_nested_metrics[[
    "model", "overall_mae", "overall_rmse", "overall_r2"
]].rename(columns={
    "model": "Model",
    "overall_mae": "MAE",
    "overall_rmse": "RMSE",
    "overall_r2": "R2",
})

final_classification_table = classification_nested_metrics[[
    "model", "oof_roc_auc", "precision_at_0_5", "recall_at_0_5",
    "f1_at_0_5", "balanced_accuracy_at_0_5",
]].rename(columns={
    "model": "Model",
    "oof_roc_auc": "ROC_AUC",
    "precision_at_0_5": "Precision_at_0_5",
    "recall_at_0_5": "Recall_at_0_5",
    "f1_at_0_5": "F1_at_0_5",
    "balanced_accuracy_at_0_5": "Balanced_Accuracy_at_0_5",
})

final_threshold_table = threshold_summary[[
    "threshold_type", "precision", "recall", "f1", "f2", "FP", "FN",
    "false_positive_rate", "miss_rate", "roc_auc",
]].copy()

display(final_regression_table)
display(final_classification_table)
display(final_threshold_table)

## 34. 生成最终评估图

所有图都使用跨受试者 outer OOF 结果，而不是训练集预测。图像保存到 `results/resume_final/figures`。

In [ ]:
# 回归 MAE 对比
plt.figure(figsize=(8, 5), dpi=150)
plt.bar(regression_nested_metrics["model"], regression_nested_metrics["overall_mae"])
plt.ylabel("OOF MAE")
plt.title("Regression Model MAE Comparison - Cross-subject OOF")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "regression_model_mae_comparison.png")
plt.show()

# 回归 R² 对比
plt.figure(figsize=(8, 5), dpi=150)
plt.bar(regression_nested_metrics["model"], regression_nested_metrics["overall_r2"])
plt.ylabel("OOF R²")
plt.title("Regression Model R² Comparison - Cross-subject OOF")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "regression_model_r2_comparison.png")
plt.show()

# 最佳回归真实值 vs 预测值
best_reg_oof = regression_outer_predictions[
    regression_outer_predictions["model"] == best_regression_model
].copy()
plt.figure(figsize=(6, 5), dpi=150)
plt.scatter(best_reg_oof["true_rpe"], best_reg_oof["pred_rpe"], s=14, alpha=0.45)
plt.plot([6, 19], [6, 19], "--", color="black", label="y=x")
plt.xlabel("True RPE")
plt.ylabel("Predicted RPE")
plt.title(f"Final Regression OOF: {best_regression_model}")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "final_regression_true_vs_pred.png")
plt.show()

## 35. 生成分类 ROC、AUC 和阈值图

ROC 曲线使用每个模型完整的 outer OOF probability。阈值 trade-off 图使用每个 outer train 内部得到的阈值曲线均值，不偷看 outer validation 标签选阈值。

In [ ]:
# 分类 AUC 对比
plt.figure(figsize=(7, 5), dpi=150)
plt.bar(classification_nested_metrics["model"], classification_nested_metrics["oof_roc_auc"])
plt.ylabel("OOF ROC-AUC")
plt.title("Classification AUC Comparison - Cross-subject OOF")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "classification_auc_comparison.png")
plt.show()

# 三个模型的 OOF ROC 曲线
plt.figure(figsize=(7, 5), dpi=150)
for model_name in classification_outer_probabilities["model"].unique():
    model_oof = classification_outer_probabilities[
        classification_outer_probabilities["model"] == model_name
    ]
    fpr, tpr, _ = roc_curve(
        model_oof["true_high_fatigue"],
        model_oof["pred_high_fatigue_probability"],
    )
    auc_value = roc_auc_score(
        model_oof["true_high_fatigue"],
        model_oof["pred_high_fatigue_probability"],
    )
    plt.plot(fpr, tpr, label=f"{model_name} AUC={auc_value:.3f}")
plt.plot([0, 1], [0, 1], "--", color="black")
plt.xlabel("False Positive Rate")
plt.ylabel("Recall / True Positive Rate")
plt.title("Classification ROC Curves - Cross-subject OOF")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "classification_roc_curves.png")
plt.show()

# 训练侧阈值 trade-off
mean_threshold_curve = threshold_curve_final.groupby("threshold")[[
    "precision", "recall", "f2"
]].mean().reset_index()
plt.figure(figsize=(7, 5), dpi=150)
for metric_name in ["precision", "recall", "f2"]:
    plt.plot(mean_threshold_curve["threshold"], mean_threshold_curve[metric_name], label=metric_name)
plt.xlabel("Probability Threshold")
plt.ylabel("Score")
plt.title("F2 Threshold Trade-off - Inner OOF Only")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "threshold_tradeoff.png")
plt.show()

## 36. 生成默认阈值和优化阈值的混淆矩阵

混淆矩阵中的 FN 表示真实 `RPE>=17`，模型却没有发出高疲劳判断。它是本实验性预警任务最关注的错误，但降低 FN 必须同时诚实报告新增 FP。

In [ ]:
for title, pred_col, filename in [
    ("Default threshold = 0.5", "pred_default_05", "final_confusion_matrix_default.png"),
    ("F2 optimized threshold", "pred_optimized", "final_confusion_matrix_optimized.png"),
]:
    cm = confusion_matrix(
        threshold_outer_results["true_high_fatigue"],
        threshold_outer_results[pred_col],
        labels=[0, 1],
    )
    plt.figure(figsize=(5, 5), dpi=150)
    plt.imshow(cm, cmap="Blues")
    plt.title(title + " - Cross-subject OOF")
    plt.xticks([0, 1], ["Pred 0", "Pred 1"])
    plt.yticks([0, 1], ["True 0", "True 1"])
    for row in range(2):
        for column in range(2):
            plt.text(column, row, str(cm[row, column]), ha="center", va="center", fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename)
    plt.show()

## 37. 合并最佳回归 OOF 与最佳分类/阈值 OOF

最终 Demo 数据只使用跨受试者 OOF 预测：当前样本所在 subject 没有参与产生该条预测的模型训练。

按照 `subject + task + exercise_folder + window_end_sec + true RPE` 一对一合并，并验证 1310 行、34 名受试者、无重复、无关键缺失值。

In [ ]:
best_regression_oof = regression_outer_predictions[
    regression_outer_predictions["model"] == best_regression_model
].copy()

demo_predictions = threshold_outer_results.merge(
    best_regression_oof[[
        "subject", "task", "exercise_folder", "window_end_sec",
        "true_rpe", "pred_rpe", "absolute_error",
    ]],
    left_on=["subject", "task", "exercise_folder", "window_end_sec", "rpe"],
    right_on=["subject", "task", "exercise_folder", "window_end_sec", "true_rpe"],
    how="inner",
    validate="one_to_one",
)

demo_predictions["regression_model"] = best_regression_model
demo_predictions["classification_model"] = best_classification_model
demo_predictions = demo_predictions[[
    "subject", "task", "exercise_folder", "window_end_sec",
    "true_rpe", "pred_rpe", "absolute_error",
    "true_high_fatigue", "pred_high_fatigue_probability",
    "pred_default_05", "pred_optimized", "selected_threshold",
    "outer_fold", "regression_model", "classification_model",
]].sort_values(["subject", "task", "window_end_sec"]).reset_index(drop=True)

critical_columns = [
    "true_rpe", "pred_rpe", "pred_high_fatigue_probability", "selected_threshold"
]
assert len(demo_predictions) == 1310
assert demo_predictions["subject"].nunique() == 34
assert demo_predictions.duplicated(["subject", "task", "window_end_sec"]).sum() == 0
assert demo_predictions[critical_columns].isna().sum().sum() == 0

demo_predictions.to_csv(DEMO_FINAL_CSV, index=False, encoding="utf-8-sig")
print("Saved:", DEMO_FINAL_CSV)
print("shape:", demo_predictions.shape)
display(demo_predictions.head(10))

## 38. 在 Notebook 中生成完整 Streamlit Demo 代码

下面不是调用已有 App，而是把 Streamlit 源码直接放在当前 Notebook 中并写出 `streamlit_app.py`。

Demo 展示历史公开数据的 OOF prediction，不连接真实传感器，也不是医疗系统。它包含 subject/task 联动、单窗口、滑块历史回放、RPE 趋势和高疲劳概率/阈值趋势。

In [ ]:
streamlit_source = f'''from pathlib import Path
import pandas as pd
import streamlit as st

PROJECT_DIR = Path(__file__).resolve().parent
DATA_PATH = PROJECT_DIR / "data_processed" / "demo_predictions_final.csv"

@st.cache_data
def load_data():
    return pd.read_csv(DATA_PATH).sort_values(["subject", "task", "window_end_sec"])

def subject_key(value):
    return int(str(value).split("_")[1])

def fatigue_level(pred_rpe):
    if pred_rpe < 13:
        return "Low"
    if pred_rpe < 17:
        return "Moderate-High"
    return "High"

st.set_page_config(page_title="Wearable IMU Fatigue Demo", layout="wide")
df_demo = load_data()

st.title("基于可穿戴 IMU 的运动疲劳评估与高疲劳预警 Demo")
st.caption("Upper-arm ACC + Gyroscope | 100 Hz | 10 s Window | Random Forest / XGBoost / Linear Models | Cross-subject OOF")
st.write("本项目使用上臂 IMU 的加速度计和陀螺仪信号，每 10 秒提取时域特征，估计 Borg RPE，并探索高疲劳状态预警。")
st.info("本 Demo 仅用于机器学习与可穿戴运动疲劳研究展示，不用于医学诊断或临床决策。")
st.write("使用历史公开数据的 OOF prediction 进行模拟实时回放。当前页面不连接真实传感器。")
st.write("Final Regression Model: **{best_regression_model}**")
st.write("Final Classification Model: **{best_classification_model}**")

subject = st.sidebar.selectbox("Subject", sorted(df_demo["subject"].unique(), key=subject_key))
subject_df = df_demo[df_demo["subject"] == subject]
task = st.sidebar.selectbox("Task", sorted(subject_df["task"].unique()))
task_df = subject_df[subject_df["task"] == task].sort_values("window_end_sec")
mode = st.sidebar.radio("模式", ["单窗口查看", "历史回放"])
windows = task_df["window_end_sec"].astype(int).tolist()

if mode == "单窗口查看":
    current_window = st.sidebar.selectbox("Window", windows, format_func=lambda x: f"{{x}} s")
else:
    position = st.sidebar.slider("当前回放到第几个窗口", 1, len(windows), len(windows))
    current_window = windows[position - 1]

current_row = task_df[task_df["window_end_sec"] == current_window].iloc[0]
visible_df = task_df[task_df["window_end_sec"] <= current_window] if mode == "历史回放" else task_df

st.subheader("当前窗口")
st.write(f"`{{subject}}` | `{{task}}` | `{{current_row.exercise_folder}}` | end={{int(current_window)}} s")
c1, c2, c3, c4 = st.columns(4)
c1.metric("True Borg RPE", f"{{current_row.true_rpe:.0f}}")
c2.metric("Predicted Borg RPE", f"{{current_row.pred_rpe:.2f}}")
c3.metric("Absolute Error", f"{{current_row.absolute_error:.2f}}")
c4.metric("High-fatigue Probability", f"{{current_row.pred_high_fatigue_probability:.3f}}")

st.write(f"Demo fatigue level: **{{fatigue_level(current_row.pred_rpe)}}**（仅为展示分级，不是医学标准）")
st.write(f"Current warning threshold: `{{current_row.selected_threshold:.3f}}`")
warning = current_row.pred_high_fatigue_probability >= current_row.selected_threshold
if warning:
    st.warning("实验性高疲劳提醒")
else:
    st.success("当前未触发高疲劳提醒")

left, right = st.columns(2)
with left:
    st.write("True High Fatigue (RPE>=17):", int(current_row.true_high_fatigue))
with right:
    st.write("Predicted Warning:", "Yes" if warning else "No")

st.subheader("True RPE vs Predicted RPE")
st.line_chart(visible_df.set_index("window_end_sec")[["true_rpe", "pred_rpe"]])
st.subheader("High-fatigue Probability vs Selected Threshold")
st.line_chart(visible_df.set_index("window_end_sec")[["pred_high_fatigue_probability", "selected_threshold"]])

with st.expander("为什么阈值不是 0.5？"):
    st.write("模型超参数在训练受试者内部选择；随后使用训练侧 inner OOF probability，以 F2 为目标选择实验性 warning threshold。降低阈值可减少高疲劳漏识别，但会明显增加误报。它不是医学标准。")

with st.expander("项目方法"):
    st.write("Upper-arm IMU → ACC + Gyroscope → 100 Hz → 10-second non-overlap window → 56 time-domain features → subject-level Nested GroupKFold → RPE regression + high-fatigue classification → experimental warning")

with st.expander("56 个特征是怎么来的？"):
    st.write("acc_x/y/z 与 gyr_x/y/z 共 6 通道，每通道提取 mean、std、min、max、range、rms、median，共 42 个；acc_mag 与 gyr_mag 各提取 7 个，共 14 个；总计 56。")

with st.expander("模型实验结果"):
    st.write("最佳回归模型：{best_regression_model}；最佳分类模型：{best_classification_model}。所有展示均来自跨受试者 OOF prediction。分类判别能力有限，降低阈值提高 Recall 的同时会增加大量 False Positive。")

with st.expander("项目局限性"):
    st.write("当前只使用单 Upper-arm IMU 和简单时域特征；RPE 是主观量表；跨受试者差异明显；分类 AUC 有限；降低预警阈值会增加误报。未来可研究多位置 IMU、PPG、EMG、个体基线归一化和更丰富时序特征，但当前项目未实现。")

with st.expander("这个 Demo 想说明什么？"):
    st.write("重点是完成可穿戴信号 → 时间窗口 → 特征工程 → 机器学习 → 跨受试者验证 → 疲劳评估 → 预警展示的完整原型，而不是构建临床级预警器。")
'''

streamlit_path = PROJECT_DIR / "streamlit_app.py"
streamlit_path.write_text(streamlit_source, encoding="utf-8")
print("Streamlit app written to:", streamlit_path)
print("启动命令: streamlit run streamlit_app.py")

## 39. 保存最终模型摘要与可追溯数字

简历或面试中使用的每个核心数字都应能从 CSV 找到。这里生成 `final_model_summary.csv`，并把最佳模型、核心指标和 Demo 行数集中保存。

In [ ]:
final_model_summary = pd.DataFrame([
    {"item": "best_regression_model", "value": best_regression_model},
    {"item": "best_classification_model", "value": best_classification_model},
    {"item": "best_regression_mae", "value": best_regression_row["overall_mae"]},
    {"item": "best_regression_rmse", "value": best_regression_row["overall_rmse"]},
    {"item": "best_regression_r2", "value": best_regression_row["overall_r2"]},
    {"item": "best_classification_auc", "value": best_classification_row["oof_roc_auc"]},
    {"item": "default_recall", "value": default_metrics["recall"]},
    {"item": "optimized_recall", "value": optimized_metrics["recall"]},
    {"item": "default_FP", "value": default_metrics["FP"]},
    {"item": "optimized_FP", "value": optimized_metrics["FP"]},
    {"item": "demo_rows", "value": len(demo_predictions)},
])
final_model_summary.to_csv(RESULTS_DIR / "final_model_summary.csv", index=False, encoding="utf-8-sig")
display(final_model_summary)

## 40. 生成最终实验报告文本

这份报告把数据概况、baseline、Nested 调参、最佳参数与阈值权衡汇总在一起。结论必须如实反映跨受试者性能，不能因为结果一般而删除受试者、改标签或改成随机窗口划分。

In [ ]:
high_count = int((y_cls == 1).sum())
non_high_count = int((y_cls == 0).sum())

report_text = f'''最终实验报告

一、数据概况
subject 数: {groups.nunique()}
有效 window 数: {len(df)}
特征数: {X.shape[1]}
RPE 范围: {y_reg.min()} - {y_reg.max()}
high_fatigue 数量: {high_count}
non_high_fatigue 数量: {non_high_count}

二、回归 baseline 结果
{regression_baseline_comparison.to_string(index=False)}

三、回归 Nested GroupKFold 结果
{regression_nested_metrics.to_string(index=False)}

四、最佳回归模型
Best Regression Model = {best_regression_model}
MAE = {best_regression_row['overall_mae']:.6f}
RMSE = {best_regression_row['overall_rmse']:.6f}
R2 = {best_regression_row['overall_r2']:.6f}

五、分类 baseline 结果
{classification_baseline_comparison.to_string(index=False)}

六、分类 Nested GroupKFold 结果
{classification_nested_metrics.to_string(index=False)}

七、最佳分类模型
Best Classification Model = {best_classification_model}
ROC-AUC = {best_classification_row['oof_roc_auc']:.6f}

八、Threshold 优化前后
{threshold_summary.to_string(index=False)}

九、方法结论
回归和分类都采用 subject-level Nested GroupKFold：外层 5 折评估未见受试者，内层 4 折选择超参数。RandomizedSearchCV 只是部分模型在内层的参数搜索方法，不等于独立的验证方案。

十、局限
多模型比较与超参数调优未带来显著性能提升，说明当前性能瓶颈可能更多来自单一 Upper-arm IMU、简单时域特征、RPE 主观性及明显的跨个体差异，而不是单纯的模型选择问题。F2 阈值显著提高 Recall 的同时也增加大量误报。
'''

(PROJECT_DIR / "final_experiment_report.txt").write_text(report_text, encoding="utf-8")
print(report_text)

## 41. 最终自动验收

最后统一检查：所有模型 OOF 行数、无 subject 泄漏、关键 CSV/图/App 存在、Demo 关键列无缺失。只有这些检查全部通过，才算从原始数据到最终 Demo 的流程完整复现。

In [ ]:
required_files = [
    FATIGUE_FEATURES_CSV,
    EXCLUDED_WINDOWS_CSV,
    RESULTS_DIR / "regression_baseline_comparison.csv",
    RESULTS_DIR / "regression_nested_metrics.csv",
    RESULTS_DIR / "regression_best_params.csv",
    RESULTS_DIR / "regression_outer_predictions.csv",
    RESULTS_DIR / "classification_baseline_comparison.csv",
    RESULTS_DIR / "classification_nested_metrics.csv",
    RESULTS_DIR / "classification_best_params.csv",
    RESULTS_DIR / "classification_outer_probabilities.csv",
    RESULTS_DIR / "threshold_summary.csv",
    RESULTS_DIR / "threshold_outer_results.csv",
    RESULTS_DIR / "final_model_summary.csv",
    DEMO_FINAL_CSV,
    PROJECT_DIR / "streamlit_app.py",
]

missing_files = [str(path) for path in required_files if not path.exists()]
assert not missing_files, f"缺少输出文件: {missing_files}"
assert len(best_regression_oof) == 1310
assert len(threshold_outer_results) == 1310
assert len(demo_predictions) == 1310
assert demo_predictions["subject"].nunique() == 34
assert demo_predictions[[
    "true_rpe", "pred_rpe", "pred_high_fatigue_probability", "selected_threshold"
]].isna().sum().sum() == 0

print("全部自动验收通过。")
print("最佳回归模型:", best_regression_model)
print("回归 MAE / RMSE / R²:",
      best_regression_row["overall_mae"],
      best_regression_row["overall_rmse"],
      best_regression_row["overall_r2"])
print("最佳分类模型:", best_classification_model)
print("分类 ROC-AUC:", best_classification_row["oof_roc_auc"])
print("默认 Recall -> 优化 Recall:", default_metrics["recall"], "->", optimized_metrics["recall"])
print("默认 FP -> 优化 FP:", int(default_metrics["FP"]), "->", int(optimized_metrics["FP"]))
print("启动 Demo: streamlit run streamlit_app.py")

## 42. 你需要记住的最终流程

回归和分类的正式实验结构完全一致：

```text
原始数据
  → 100 Hz Upper-arm ACC + Gyroscope
  → 10 秒非重叠窗口
  → 56 个时域特征
  → subject-level outer GroupKFold（5 折，评估）
      → 每个 outer train 内再做 inner GroupKFold（4 折，调参）
          → Ridge / Logistic：GridSearchCV
          → RF / XGBoost / SVR：RandomizedSearchCV（20 次）
      → 用最佳参数拟合完整 outer train
      → 预测未见 outer validation subjects
  → 拼接 5 折得到 1310 条 OOF prediction
```

分类的最佳模型确定后，才额外进行 F2 warning threshold optimization；阈值同样只在 outer train 的 inner OOF probability 上选择。

所以，`RandomizedSearchCV` 回答的是“内层怎样尝试参数”，`Nested GroupKFold` 回答的是“调参和最终评估怎样隔离”。这两个概念不是二选一。

真实结果表明项目存在一定跨受试者信号，但性能仍有限。阈值降低可以显著减少高疲劳漏识别，同时会付出大量误报代价。该结论适合机器学习研究 Demo，不代表医学诊断能力。